# **Task 5: Fine-tune a transformer model (BERT/DistilBERT)**

**Setup and Installations**

In [3]:
!pip install transformers datasets evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=0e7bc281d794cf648f7d8f928945ef03a26f0d3025a4c7b79b67c7e6e98d91ee
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:

import datasets
import transformers

datasets.disable_progress_bar()
transformers.utils.logging.disable_progress_bar()

In [ ]:
from tqdm import tqdm

for i in tqdm(range(100), disable=True):
    pass


**Dataset Selection, Data Preprocessing & Model Setup**

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)
import evaluate
import numpy as np

from datasets import load_dataset

# Load the Dataset
dataset=load_dataset("lhoestq/conll2003")
chunk_labels=[
    "O", "B-ADJP", "I-ADJP", "B-ADVP", "I-ADVP", "B-CONJP", "I-CONJP",
    "B-INTJ", "I-INTJ", "B-LST", "I-LST", "B-NP", "I-NP", "B-PP", "I-PP",
    "B-PRT", "I-PRT", "B-SBAR", "I-SBAR", "B-UCP", "I-UCP", "B-VP", "I-VP"
]

id2label={i: label for i, label in enumerate(chunk_labels)}
label2id={label: i for i, label in enumerate(chunk_labels)}
print("Labels successfully mapped!")

# Load Tokenizer & DistilBERT
model_checkpoint="distilbert-base-cased"
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)
model=AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_labels),
    id2label=id2label,
    label2id=label2id
)

# Data Preprocessing
def tokenize_and_align_labels(examples):
    tokenized_inputs=tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    labels=[]
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids=tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx=None
        label_ids=[]
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx=word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"]=labels
    return tokenized_inputs
tokenized_datasets=dataset.map(tokenize_and_align_labels, batched=True)


Labels successfully mapped!


DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Training & Evaluation**

In [ ]:
import evaluate
# Load the seqeval metric
seqeval=evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions=np.argmax(predictions, axis=2)
    true_predictions=[
        [chunk_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels=[
        [chunk_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results=seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

args=TrainingArguments(
    output_dir="./distilbert-chunking",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer=Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
trainer.train()

**Inference**

In [ ]:
from transformers import pipeline
chunking_pipeline=pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

text="John works at Google in California"
predictions=chunking_pipeline(text)

print("Input:", text)
for entity in predictions:
    print(f"Word/Phrase: {entity['word']} | Tag: {entity['entity_group']} | Score: {entity['score']:.4f}")


**Comparison (POS vs Chunking)**

 * POS Tagging (Word-Level Tagging): It involves assigning grammatical labelsto each individual word. (Example: John – Noun, works – Verb). This task is less complex for machines since it requires only a small context window to assign POS tags.

 * Chunking (Phrase-level grouping): Here, words that are related are grouped together into phrases (e.g., “The quick brown fox” = Noun Phrase). This task has a medium level of difficulty since, apart from recognizing word types, the model should also recognize phrases’ boundaries using the BIO tag set.

**Report**

**The Core Difference**

Even though both tasks fall under token classification, they differ greatly when it comes to their purposes:

In POS tagging, one deals with high detail: "What grammatical function does this particular word play?" This approach focuses on individual tokens, where 'Google' becomes a Proper Noun and 'in' a Preposition.

In chunking, one deals with structure: "How do words combine into meaning units?" This task focuses on spans of text: it sees "Google in California" as being part of phrase construction with the help of BIO (beginning, inside, outside) tags for marking up phrase boundaries.

Challenges faced: The Tokenization Labeling Problem

The major difficulty that comes up in utilizing BERT-type models for token labeling is related to the mismatch between the word in question and the tokens generated through the tokenization process carried out by the model. The model uses sub-word tokenization, such as WordPiece. For instance, "tokenization" can be tokenized into "token," "##iza," and "##tion." If there is only one label linked to the whole word, then the problem is based on the lack of consistency. The approach that was used for solving this issue is masking, whereby the real label is given to the first token, "token," while the other two tokens are labeled "-100." The label is used as an ignore index in PyTorch's CrossEntropyLoss.

Observations and insights

The use of DistilBERT resulted in huge speedups during training without losing accuracy. The performance of the model was measured by using seqeval, a metric that takes into account the entire sequence as opposed to looking at each token individually. This gives us the Precision, Recall, and F1 Score. The model could easily grasp POS tags because of their locality, but took longer to recognize the correct phrase boundaries required for Chunking.